In [ ]:
#Columns used
#Furnace Temp, Carbon Content, Oxygen Levels, Silicon Content, Manganese Content, Sulfur Content, Phosphorus Content, Scrap Ratio, Power input, Tap Weight
#Range
# 1550-1650, 0.02-0.8, 0.5-3.0, 0.1-0.4, 0.3-1.5, 0-0.05, 0-0.04,20-80, 300-600, 100-300 
#Output steel grade

import numpy as np
import pandas as pd

n_hours = 24

data = {
    'Furnace_Temp': np.random.uniform(1550, 1650, n_hours),
    'Carbon_Content': np.random.uniform(0.02, 0.8, n_hours),
    'Oxygen Levels': np.random.uniform(0.5, 3.0, n_hours),
    'Silicon Content': np.random.uniform(0.1, 0.4, n_hours),
    'Manganese Content': np.random.uniform(0.3, 1.5, n_hours),
    'Sulfur Content': np.random.uniform(0.0, 0.05, n_hours),
    'Phosphorus Content': np.random.uniform(0.0, 0.04, n_hours),
    'Scrap Ratio': np.random.uniform(20, 80, n_hours),
    'Power input': np.random.uniform(300, 600, n_hours),
    'Tap Weight': np.random.uniform(100, 300, n_hours),
}

df = pd.DataFrame(data)

df.loc[10, 'Sulfur Content'] = 0.12    # way above normal max of 0.05
df.loc[18, 'Furnace_Temp'] = 1480      # too low, below normal min of 1550

df['Timestamp'] = pd.date_range(start='2024-01-01', periods=n_hours, freq='h')

df.to_excel('steel_readings.xlsx', index=False)
print("Excel file saved!")
#print(df.head())



Excel file saved!
(24, 11)


In [9]:
import pandas as pd
import numpy as np

df_input = pd.read_excel('steel_readings.xlsx')
print(df_input.shape)

#to read anomalies, i can add set ranges and check each row within the range by creating a lower bound array and an 
#upper bound array and check if every element is within the range

def detect_anomalies(df):
    anomalies = []

    data_columns = [col for col in df_input.columns if col != 'Timestamp']
    lower_values = np.array([1550.0, 0.02, 0.5, 0.1, 0.3, 0, 0, 20, 300, 100])
    upper_values = np.array([1650.0, 0.8, 3, 0.4, 1.5, 0.05, 0.04, 80, 600, 300])

    for column, lower_value, upper_value in zip(data_columns, lower_values, upper_values):
        low_val = df[df[column] < lower_value]
        upper_val = df[df[column] > upper_value]

        if len(low_val) > 0:
            low_val['issue'] = f'{column} value too low'
            anomalies.append(low_val)
        
        if len(upper_val) > 0:
            upper_val['issue'] = f'{column} value too high'
            anomalies.append(upper_val)

    return pd.concat(anomalies) if anomalies else pd.DataFrame()
            


results = detect_anomalies(df_input)
print(results)

(24, 11)
    Furnace_Temp  Carbon_Content  Oxygen Levels  Silicon Content  \
18   1480.000000        0.387217       0.713841         0.127315   
10   1637.046795        0.105389       1.189269         0.233722   

    Manganese Content  Sulfur Content  Phosphorus Content  Scrap Ratio  \
18           0.448291        0.036243            0.038441    56.985062   
10           0.947918        0.120000            0.012812    75.372251   

    Power input  Tap Weight           Timestamp                          issue  
18   587.118455  292.015128 2024-01-01 18:00:00     Furnace_Temp value too low  
10   554.189596  285.952566 2024-01-01 10:00:00  Sulfur Content value too high  


In [19]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import anthropic

load_dotenv()

apiKey = os.getenv("ANTHROPIC_API_KEY")

df_input = pd.read_excel("steel_readings.xlsx");

def detect_anomalies(df):
    anomalies = []

    columns = [col for col in df_input.columns if col != "Timestamp"]
    lower_values = np.array([1550.0, 0.02, 0.5, 0.1, 0.3, 0, 0, 20, 300, 100])
    upper_values = np.array([1650.0, 0.8, 3, 0.4, 1.5, 0.05, 0.04, 80, 600, 300])

    for column, lower_value, upper_value in zip(columns, lower_values, upper_values):
        low_val = df[df[column] < lower_value]
        high_val = df[df[column] > upper_value]

        if len(low_val) > 0:
            low_val = low_val.copy()
            low_val["issue"] = f"{column} has a lower than expected value when it should be between {lower_value} and {upper_value} at {low_val['Timestamp'].values[0]}"
            anomalies.append(low_val)
        if len(high_val) > 0:
            high_val = high_val.copy()
            high_val["issue"] = f"{column} has a higher than expected value when it should be between {lower_value} and {upper_value} at {high_val['Timestamp'].values[0]}"
            anomalies.append(high_val)

    return pd.concat(anomalies) if anomalies else pd.DataFrame()

results = detect_anomalies(df_input)


def get_recommendations(input):
    client = anthropic.Anthropic()

    message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system="You are a blast furnace control room operator in the steel manufacturing business.",
    messages=[
        {"role": "user", 
         "content": f"Based on the data provided here {input}, what would be your recommendation for the blast furnace control room operator. Answer in a user friendly manner where your recommendation will be read by the plant manager"}
    ]
    )
    print(message.content[0].text)


get_recommendations(results.to_string())
    

# BLAST FURNACE OPERATIONAL ALERT & RECOMMENDATIONS

**TO: Plant Manager**
**FROM: Control Room Analysis**
**DATE: 2024-01-01**

---

## **SUMMARY OF ISSUES IDENTIFIED**

We have two operational concerns that require immediate attention:

### **Issue #1: LOW FURNACE TEMPERATURE (18:00 Hours)**
- **Current Temperature:** 1480°C
- **Target Range:** 1550–1650°C
- **Status:** ⚠️ **70°C BELOW MINIMUM**

### **Issue #2: ELEVATED SULFUR CONTENT (10:00 Hours)**
- **Current Level:** 0.120% 
- **Maximum Acceptable:** 0.05%
- **Status:** ⚠️ **2.4x ABOVE SPECIFICATION**

---

## **IMMEDIATE RECOMMENDATIONS**

### **For the Low Temperature Issue:**
1. **Increase power input gradually** (currently at 587 MW) – target 610-630 MW
2. **Improve burden distribution** to enhance heat transfer efficiency
3. **Check fuel quality** – verify coke reactivity and consumption rates
4. **Inspect blast temperature and volume** settings
5. **Monitor in 30-minute intervals** until temperature returns to 1550°C minim

In [14]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import anthropic
from dotenv import load_dotenv
import os

def generate_excel():
    n_hours = 24

    data = {
        "HMT": np.random.uniform(1440, 1510, n_hours),
        "Slag_Al2O3": np.random.uniform(19.5, 21.5, n_hours),
        "W/V": np.random.uniform(48000, 56000, n_hours)    
    }

    df = pd.DataFrame(data)

    df["Timestamp"] = df['Timestamp'] = pd.date_range(start='2026-09-22', periods=n_hours, freq='h')

    df.to_excel('steel_readings_2.xlsx', index=False)

#generate_excel()


def generate_readings():
    rng = np.random.default_rng(seed=42)
    num_rows = 250

    data = {
        "HMT": rng.integers(low=1440, high=1510, size=num_rows),
        "Slag_Al2O3": rng.uniform(low=19.5, high=21.5, size=num_rows),
        "W/V": rng.integers(low=48000, high=56000, size=num_rows)
    }

    df = pd.DataFrame(data)
    flag_rows = pd.DataFrame({
        "HMT": np.concatenate([
            rng.integers(low=1420, high=1440, size=25),  # too low
            rng.integers(low=1510, high=1530, size=25)   # too high
        ]),
        "Slag_Al2O3": rng.uniform(low=19.5, high=21.5, size=50),
        "W/V": rng.integers(low=48000, high=56000, size=50)
    })

    df = pd.concat([df, flag_rows], ignore_index=True)

    conditions = [
    ((df["HMT"] > 1510) | (df["HMT"] < 1440) | (df["Slag_Al2O3"] > 21.5) | (df["Slag_Al2O3"] < 19.5)),
    ((((df["HMT"] > 1440) & (df["HMT"] < 1450)) | ((df["HMT"] > 1500) & (df["HMT"] < 1510))) | (((df["Slag_Al2O3"] > 19.5) & (df["Slag_Al2O3"] < 19.8)) | ((df["Slag_Al2O3"] > 21.2) & (df["Slag_Al2O3"] < 21.5)))) 
    ]

    output = ["flag_issue", "hold"]

    df["Output"] = np.select(conditions, output, default="increase_WV")
    return df



def train_model():
    readings = generate_readings()

    X_train, X_test, y_train, y_test = train_test_split(readings[["HMT", "Slag_Al2O3"]], readings["Output"], test_size=0.2, train_size = 0.8, shuffle=True)

    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)

    print(model.score(X_test, y_test))

    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    return model

model = train_model()
print(type(model))


def analyze_readings(model):

    df = pd.read_excel('steel_readings_2.xlsx')
    df['Predicted_Output'] = model.predict(df[['HMT', 'Slag_Al2O3']])

    load_dotenv()

    apiKey = os.getenv("ANTHROPIC_API_KEY")

    client = anthropic.Anthropic()

    message = client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens = 1024,
        system = "You are blast furnace professional",
        messages=[
            {"role": "user", "content": f"based on the \n{df.to_string()} provided, can you provide the recommendation for the blast furnace operations. The response should be in a user friendy way"}
        ]
    )

    print(message.content[0].text)

analyze_readings(model)

1.0
              precision    recall  f1-score   support

  flag_issue       1.00      1.00      1.00        10
        hold       1.00      1.00      1.00        28
 increase_WV       1.00      1.00      1.00        22

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

<class 'sklearn.tree._classes.DecisionTreeClassifier'>
# Blast Furnace Operations Recommendation Report
**Date: September 22, 2026**

---

## 📊 Executive Summary

Based on the analysis of 24 hours of operational data, your blast furnace is showing **mixed performance** with alternating patterns of stability and adjustment needs.

---

## 🎯 Key Findings

| Metric | Status | Details |
|--------|--------|---------|
| **HMT (Hearth Metal Temp)** | ✅ Stable | Range: 1,442-1,507°C - Within acceptable parameters |
| **Slag Al₂O₃ Content** | ⚠️ Moderate | Range: 19.6-21.5% - Some variability noted |
| **W/V Ratio** 